In [84]:
# conda activate anndata

import os
import sys
import pandas as pd
import anndata as ad
import concurrent.futures

sys.path.append("/mnt/lareaulab/reliscu/code")

from junction2psi import *

os.chdir("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA_MOp_VISp")

Here I make pseudobulk splice junction data, using the same cells in each sample as were used to generate the gene-level pseudobulk. These pseudobulk SJ counts will be used to calculate pseudobulk PSIs for each exon.

In [ ]:
pseudobulk_data = "yao_2021_ACA_MOp_VISp_STAR_SyntheticDatasets/SyntheticDataset1_10pcntCells_20SD_200samples"
pseudobulk_meta = pd.read_csv("data/SyntheticDatasets/SyntheticDataset1_10pcntCells_20SD_200samples_legend_02-37-49.csv")

In [85]:
intron_file = "/mnt/lareaulab/reliscu/data/GENCODE/GRCm39/psix_annotation/intron_file.tab.gz"
sdata = ad.read_h5ad("data/yao_2021_ACA_MOp_VISp_STAR_SJ_counts.h5ad")

intron_table = read_intron_file(intron_file)

In [12]:
# Track where rows of SJ count data end up after merging with annotations
# Some junctions (rows) will be duplicated becauase a junction can be associated with multiple exons
sdata.var['row'] = range(sdata.var.shape[0])
sj_table = pd.merge(intron_table, sdata.var, left_on="intron", right_index=True, how="inner")

sdata_filt = sdata[:, sj_table['row']].copy() 
sdata_filt.var = sj_table

/mnt/lareaulab/reliscu/anaconda3/envs/anndata/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [13]:
# Now create SJ pseudobulk: 
# Get the cells comprising each pseudobulk gene expression sample, then sum up SJ counts over those cells

X = sdata_filt.X.toarray()

count_mat = pseudobulk_meta.iloc[:, 2:].astype(int).to_numpy() # shape: cells × # pseudobulk samples 
cell_ids = pseudobulk_meta['Cell.name'].astype(str).tolist()

pseudo_list = []
for i in range(count_mat.shape[1]):
    cell_tally = count_mat[:,i]
    rep_labels = np.repeat(cell_ids, cell_tally) # Cells were drawn with replacement so there will be repeats
    idx = sdata_filt.obs_names.get_indexer(rep_labels)
    X = sdata_filt.X[idx, :]
    pseudo_list.append(X.sum(axis=0))

pseudo_array = np.vstack(pseudo_list).T
col_idx = [f"Sample{i+1}" for i in range(pseudo_array.shape[1])]
row_idx = list(sdata_filt.var_names)
pseudo_df = pd.DataFrame(pseudo_array, index=row_idx, columns=col_idx)

In [14]:
pseudo_df.shape

(118868, 200)

In [15]:
# Remove to jxns with 0 counts
pseudo_df_filt = pseudo_df.loc[(pseudo_df.sum(axis=1) > 0).values]

# Subset single-cell data to the same jxns:

mask = sdata_filt.var_names.isin(pseudo_df_filt.index)
sdata_refilt = sdata_filt[:, mask].copy()

In [16]:
sdata_refilt.shape

(23801, 118285)

In [17]:
pseudo_df_filt.shape

(118285, 200)

### Filter jxns, calc PSI for pseudobulk data

In [18]:
SJ_counts_table = pseudo_df_filt.copy()

In [19]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events

In [20]:
len(events)

11906

In [21]:
I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]
filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [22]:
psi.to_csv(f"data/{pseudobulk_data}_SJ_PSI.csv")
reads.to_csv(f"data/{pseudobulk_data}_SJ_counts.csv")

### Filter jxns, calc PSI for single cell data

In [23]:
SJ_counts_table = pd.DataFrame.sparse.from_spmatrix(
    sdata_refilt.X.T, columns=sdata_refilt.obs_names, index=sdata_refilt.var_names
)

In [24]:
events_i1 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I1' in x])
events_i2 = pd.Index([x[:-3] for x in SJ_counts_table.index if '_I2' in x])
events_se = pd.Index([x[:-3] for x in SJ_counts_table.index if '_SE' in x])

events = events_i1.intersection(events_i2).intersection(events_se)
i1_events = [x + '_I1' for x in events]
I1_table = SJ_counts_table.loc[i1_events]
I1_table.index = events

i2_events = [x + '_I2' for x in events]
I2_table = SJ_counts_table.loc[i2_events]
I2_table.index = events

se_events = [x + '_SE' for x in events]
SE_table = SJ_counts_table.loc[se_events]
SE_table.index = events

I1_filt = I1_table.index[I1_table.sum(axis=1) > 0]
I2_filt = I2_table.index[I2_table.sum(axis=1) > 0]
SE_filt = SE_table.index[SE_table.sum(axis=1) > 0]
filtered_events = I1_filt.intersection(I2_filt).intersection(SE_filt)

I1_table = I1_table.loc[filtered_events]
I2_table = I2_table.loc[filtered_events]
SE_table = SE_table.loc[filtered_events]

psi = ((I1_table + I2_table) /(2*SE_table + I1_table + I2_table)).fillna(0)
reads = SE_table + I1_table + I2_table

In [71]:
psi.shape

(11906, 23801)

In [ ]:
sj_table['exon'] = sj_table.index.str.split("_").str[:-1].str.join("_")
sj_table = sj_table[sj_table['exon'].isin(psi_var.index)]

In [ ]:
# Get coordinates per exon

exon_coords = sj_table.groupby("exon").apply(lambda g: pd.Series({
    "exon_start": g.loc[g.index.str.contains("I1"), "intron_last_base"].values[0] + 1,
    "exon_end":   g.loc[g.index.str.contains("I2"), "intron_first_base"].values[0] - 1,
}))

In [73]:
exon_coords = exon_coords.reindex(psi.index)

In [75]:
# Save PSI data anndata object

psi_mat = psi.to_numpy(dtype=np.float32)
obs = sdata.obs
var = exon_coords

In [76]:
adata_psi = ad.AnnData(
    X=psi_mat.T,
    obs=obs,
    var=var,
)

In [80]:
# Keep raw counts as well
reads_df = reads.T.reindex(index=adata_psi.obs_names, columns=adata_psi.var_names)
adata_psi.layers["exon_counts"] = reads_df.to_numpy(dtype=np.float32)

In [83]:
adata_psi.write(f"data/yao_2021_ACA_MOp_VISp_STAR_SJ_counts_PSI_annotated.h5ad")